In [ ]:
from unsloth import FastLanguageModel,PatchFastRL,is_bfloat16_supported
import torch,os
import re
from datasets import load_from_disk, Dataset
PatchFastRL("GRPO", FastLanguageModel)


os.environ["VLLM_USE_V1"] = '0'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/mnt/project/Qwen2.5-7B-GRPO/model/Qwen2.5-7B",
    max_seq_length=10240,
    dtype=torch.bfloat16,   # 强制使用 bfloat16
    load_in_4bit=True, 
    fast_inference = True, 
    max_lora_rank = 64,
    gpu_memory_utilization = 0.5
)
# Get PEFT model
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "gate_proj",
        "up_proj",
        "down_proj",
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_alpha=64,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


In [ ]:
for name, param in model.named_parameters():
    print(f"Parameter {name} is not quantized, dtype is {param.dtype}.")

In [ ]:
from datasets import load_from_disk
import re
from datasets import load_dataset, Dataset

# Load and prep dataset
SYSTEM_PROMPT = """
作为医学专家，请按以下格式回答：
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def get_gsm8k_questions() -> Dataset:
    data =load_from_disk("/mnt/project/Qwen2.5-1.5B-lora/data/cMedQA2/cMedQA2-instructions/train")
    # 修复列名：anwser -> answer
    data = data.rename_column('anwser', 'answer')
    # 移除不需要的列
    data = data.remove_columns(['question_id', 'answer_id'])
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        # 'answer': x['anwser']
    }) # type: ignore
    return data # type: ignore
dataset = get_gsm8k_questions()
dataset[:100]


In [ ]:
# Reward functions

import evaluate
import bert_score  # 为了添加 model2layers 信息
import numpy as np

# 只加载一次（全局初始化）
bertscore = evaluate.load("bertscore")
bert_score.utils.model2layers["/mnt/project/bert-base-chinese"] = 12  # 添加模型层数

# 文本相似性奖励函数
def similarity_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    # 提取响应文本
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    # 使用 BERTScore 批量计算
    bertscore_results = bertscore.compute(
        predictions=extracted_responses,
        references=answer,
        lang="zh",
        model_type="/mnt/project/bert-base-chinese",
        batch_size=16,
        rescale_with_baseline=False
    )

    # precisions = bertscore_results["precision"]
    # recalls = bertscore_results["recall"]
    f1s = bertscore_results["f1"]
    # 综合 reward，每项最大值是 1，三项相加最大是 3，* 2/3 缩放到 0~2 区间
    return [ f * 2.0  for f in f1s]

# 根据生成答案的长度奖励函数
def length_reward_func(completions, **kwargs) -> list[float]:
    # 提取响应文本
    responses = [completion[0]['content'] for completion in completions]
    rewards = []
    for response in responses:
        # 获取答案的字符长度
        length = len(response)
        
        # 假设我们期望答案长度在128到512之间（根据实际任务可以调整）
        if length < 64:
            rewards.append(0.0)  # 答案太短，低奖励
        elif 64 <= length <= 128:
            rewards.append(0.25)  # 合理长度，最大奖励
        elif 128 <= length <= 256:
            rewards.append(0.5)  # 合理长度，最大奖励
        elif 256 <= length <= 512:
            rewards.append(1)  # 合理长度，最大奖励
        else:
            rewards.append(0.25)  # 过长的答案稍微减分
    return rewards

# 严格格式奖励 验证XML结构的严格合规性
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

# 宽松格式奖励 验证基本XML结构的存在性
def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count
# XML结构计数奖励
def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]


In [ ]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 8, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 200,
    num_train_epochs = 2, # Set to 1 for a full training run
    save_strategy="epoch",
    max_steps = 250,
    save_steps = 250,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "../output/Qwen2.5-1.5B-lora-medical-GRPO",
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        length_reward_func,
        strict_format_reward_func,
        similarity_reward_func
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "一位45岁女性患者有长期胃食管反流病（GERD），正在服用奥美拉唑（质子泵抑制剂，PPI）多年。最近她出现手足抽搐和感觉异常，实验室检查显示血钙降低（2.0 mmol/L，正常 2.2–2.6）。她没有肾功能不全或维生素D缺乏的病史。这位患者的低钙血症最可能的原因是什么？应如何处理？"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 10240,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

In [ ]:
model.save_lora("/mnt/project/Qwen2.5-1.5B-lora/output/Qwen2.5-1.5B-lora-cMedQA/grpo_saved_lora_4.0")

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" :  "一位45岁女性患者有长期胃食管反流病（GERD），正在服用奥美拉唑（质子泵抑制剂，PPI）多年。最近她出现手足抽搐和感觉异常，实验室检查显示血钙降低（2.0 mmol/L，正常 2.2–2.6）。她没有肾功能不全或维生素D缺乏的病史。这位患者的低钙血症最可能的原因是什么？应如何处理？"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("/mnt/project/Qwen2.5-1.5B-lora/output/Qwen2.5-1.5B-lora-cMedQA/grpo_saved_lora_4.0"),
)[0].outputs[0].text

output